In [1]:
import numpy as np
import torch
import os
import pickle
import gzip
from components.other_utilities.models_to_train import ResNetPLModel
from components.FL_sim import FLSimulator, RawBroadcastProtocol
from components.other_utilities.datasets import FasterSVHN
from torchvision import transforms

In [2]:
torch.set_float32_matmul_precision('medium')

import logging
import warnings
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
logger = logging.getLogger('pytorch_lightning.utilities.rank_zero')
logger.setLevel(logging.ERROR)
warnings.filterwarnings("ignore", "LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]")
warnings.filterwarnings("ignore", "The 'train_dataloader' does")
warnings.filterwarnings("ignore", "You defined a `validation_step` but")
warnings.filterwarnings("ignore", "Starting from v1.9.0, `tensorboardX` has been")
warnings.filterwarnings("ignore", "The number of training batches")
warnings.filterwarnings("ignore", "`Trainer.fit` stopped: ")

In [3]:
print('Starting the script... dataset')

if '_dataset_' not in globals():
    _dataset_ = [
        FasterSVHN(
            root='../data/SVHN', split=s,
            transform=transforms.Compose([
                transforms.Resize(32),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.4377, 0.4438, 0.4728],
                    std=[0.1980, 0.2010, 0.1970]
                ),
            ])
        ) for s in ['train', 'test']]
dataset = _dataset_

# dataset = [torch.utils.data.Subset(d, list(range(100))) for d in dataset]
# for i in range(10):
#     for d in dataset:
#         d.dataset.labels[i]=i

Starting the script... dataset


In [4]:
print('Starting the script... actual model')
batch_size = 7_500
__epoch_count__ = 10
batch_count = np.ceil(len(dataset[0])/batch_size/2)

# *****************
path_exp = 'exp_data/save_grads_per_round'
if not os.path.exists(path_exp):
    os.makedirs(path_exp)
class SaveGradBroadcastProtocol(RawBroadcastProtocol):
    round_id = -1
    def to_server_prep_data_for_transfer(self, agent_id, grad_dict, encoder_data_sent_by_server):
        if agent_id == 0:
            self.round_id += 1
            print(f'Saving gradients for round {self.round_id}, worker {agent_id}')
        with gzip.open(os.path.join(path_exp, f'grad_round_{self.round_id}_worker_{agent_id}.pkl.gz'),
                       'wb', compresslevel=1) as f:
            pickle.dump(grad_dict, f)
        return (grad_dict, )
# *****************
model = ResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.005, )
model.load_state_dict(torch.load('../data/resnet18_svhn.pth', map_location='cpu'))
# *****************
sim = FLSimulator(
    pl_model=model, num_agents=2, communication_rounds=50, client_epochs_per_round=60,
    batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
    aggregation_method='fedavg', non_iid_sampling=False, user_logger=None)
# ****
sim.run_simulation(SaveGradBroadcastProtocol())



Starting the script... actual model

round 1/50 --------------------
  - reporting global model metrics
         - train> loss: 2.18, auc: 0.61    |    test> loss: 2.17, auc: 0.61

      > training agent 1/2


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


             + initiating broadcast
Saving gradients for round 0, worker 0
         - train> loss: 0.90, auc: 0.95    |    

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


test> loss: 1.08, auc: 0.93
      > training agent 2/2
             + initiating broadcast
         - train> loss: 0.88, auc: 0.96    |    test> loss: 1.05, auc: 0.93
Aggregating gradients using FedAvg...

round 2/50 --------------------
  - reporting global model metrics
         - train> loss: 1.61, auc: 0.87    |    test> loss: 1.55, auc: 0.89

      > training agent 1/2


...

test> loss: 1.04, auc: 0.97
      > training agent 2/2
             + initiating broadcast
         - train> loss: 0.00, auc: 1.00    |    test> loss: 1.04, auc: 0.97
Aggregating gradients using FedAvg...

round 50/50 --------------------
  - reporting global model metrics
         - train> loss: 1.31, auc: 0.94    |    test> loss: 1.38, auc: 0.92

      > training agent 1/2


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


             + initiating broadcast
Saving gradients for round 49, worker 0
         - train> loss: 0.00, auc: 1.00    |    

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


test> loss: 1.05, auc: 0.97
      > training agent 2/2
             + initiating broadcast
         - train> loss: 0.00, auc: 1.00    |    test> loss: 1.05, auc: 0.97
Aggregating gradients using FedAvg...

final global model metrics
         - train> loss: 1.32, auc: 0.94    |    test> loss: 1.38, auc: 0.92
